# 01 - Collecte des données

À lancer **rarement** (par exemple une fois par mois) : récupère la courbe BAM, la liste des emprunts obligataires AMMC et lit les notes d'opération PDF.

| Étape | Source | Résultat (cache) | Durée la 1re fois |
|---|---|---|---|
| 1 | Bank Al-Maghrib | `data/courbe_bam.csv` | quelques secondes |
| 2 | AMMC (liste + pages de détail) | `data/ammc_emissions_details.csv` | ~10 min |
| 3 | Notes d'opération PDF | `data/pdfs/` + `data/tranches.csv` | ~5 min |

Chaque étape relit son cache s'il existe. Pour tout reprendre de zéro : `rafraichir=True`.

Ensuite, passe à **`02_analyse.ipynb`**.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 250)

from courbe import get_courbe_bam, taux_bam
from ammc import collecter_emissions, extraire_emetteur
from pipeline import construire_tranches

## 1. Courbe des taux BDT (Bank Al-Maghrib)

In [ ]:
courbe = get_courbe_bam(rafraichir=True)      # True = courbe du jour
print("Date de valeur :", pd.to_datetime(courbe["date_valeur"]).iloc[0].date())
print("BDT 5 ans :", round(taux_bam(5, courbe) * 100, 3), "%  |  BDT 10 ans :", round(taux_bam(10, courbe) * 100, 3), "%")
courbe

## 2. Emprunts obligataires visés par l'AMMC

In [ ]:
base = collecter_emissions(rafraichir=False)
base["emetteur"] = base["operation"].apply(extraire_emetteur)     # noms normalisés
print(len(base), "émissions calculables |", base["pdf_note"].notna().sum(), "avec note d'opération")
base["type"].value_counts()

## 3. Lecture des notes d'opération (une ligne par tranche)

In [ ]:
tranches = construire_tranches(base, rafraichir=False)
print(len(tranches), "tranches extraites")
tranches.head(10)

### Contrôle qualité : part d'informations manquantes

In [ ]:
colonnes = ["echeance", "maturite_ans", "nature_taux", "prime_pb", "remboursement"]
(tranches[colonnes].isna().mean() * 100).round(1).rename("% manquant")

Les données sont prêtes. Passe à **`02_analyse.ipynb`**.